In [ ]:
import json
import time
import os
import re
import copy
from pathlib import Path
import kaggle_benchmarks as kbench

In [ ]:
# Helpers
def normalize_segment_text(text: str) -> str:
    text = text.strip()
    if not text:
        return text
    END_PUNCTS = ('.', '?', '!', ';', ':', '…', ',')
    if not text.endswith(END_PUNCTS):
        text += "."
    return text

In [ ]:
# Wait 30s between successful calls to avoid hitting rate limits. 
# You can adjust this value based on your LLM's rate limits and your needs. 
DELAY_BETWEEN_CALLS = 30.0  

@kbench.task(name=f"Fix Event Grouping", description=f"Gom lại segments thành events đúng theo chủ đề")
def fix_event_grouping_task(llm) -> None:

    # video transcripts source files
    transcript_dir = Path(f"./transcripts")
    base_output_dir = f"./grouped_event"

    if not transcript_dir.exists():
        print(f"❌ Không tìm thấy thư mục: {transcript_dir}. Hãy kiểm tra lại LEVEL_NUMBER!")
        return

    raw_paths = transcript_dir.glob("*_transcript.json")
    all_json_paths = sorted(set(
        [str(p) for p in raw_paths]
    ))
    total_all_files = len(all_json_paths)

    if total_all_files == 0:
        print(f"⚠️ Không tìm thấy file JSON nào trong {transcript_dir}")
        return

    # chunk file nếu muốn chạy theo phần
    part_paths = all_json_paths 
    total_part_files = len(part_paths)

    print(f"Starting fix_event_grouping_task... (Tổng {total_part_files}/{total_all_files} files)")
    print("----------------------------------------------------------------------")

    for idx, input_path in enumerate(part_paths, 1):
        filename = Path(input_path).name
        relative_path = f"transcripts/{filename}"
        output_path = os.path.join(base_output_dir, relative_path)

        if os.path.exists(output_path):
            print(f"⏩ [{idx}/{total_part_files}] [BỎ QUA] File {filename} đã xử lý trước đó.")
            continue

        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        print(f"🎬 [{idx}/{total_part_files}] Đang xử lý: {filename}")

        try:
            with open(input_path, "r", encoding="utf-8") as f:
                data = json.load(f)
        except Exception as e:
            print(f"    ❌ Lỗi đọc file: {e}. Bỏ qua file này.")
            continue

        old_events = data.get("events", [])

        # Flatten & collect segments
        flat_segments = []
        if old_events:
            for ev in old_events:
                for seg in ev.get("segments", []):
                    flat_segments.append(seg)
        else:
            flat_segments = data.get("segments", [])

        flat_segments.sort(key=lambda s: s["start"])

        total_segs = len(flat_segments)

        # Check nếu hoàn toàn không có segment nào
        if total_segs == 0:
            print("    ⏩ File không có segments/events hợp lệ. Copy nguyên trạng.")
            with open(output_path, "w", encoding="utf-8") as f:
                json.dump(data, f, ensure_ascii=False, indent=2)
            continue

        # 2. PAYLOAD BUILDER
        payload_lines = []
        for i, seg in enumerate(flat_segments):
            text = seg["text"].strip().replace('"', "'")
            payload_lines.append(f"[{i}] {text}")
        payload_text = "\n".join(payload_lines)

        prompt = "" 
        # prompt được thay đổi dựa trên topic của video, check benchmark/prompts để xem chi tiết
        # cần đảm bảo phải cung cấp 2 biến 'total_segs' và 'payload_text' trong prompt để AI biết số lượng segment và nội dung segment

        max_retries = 3
        retry_count = 0
        success = False
        new_events = None

        while retry_count < max_retries and not success:
            try:
                response_text = llm.prompt(prompt)

                # Làm sạch Regex cho Markdown codeblock
                cleaned = re.sub(r"^```(?:json)?|```$", "", response_text.strip(), flags=re.MULTILINE | re.IGNORECASE).strip()
                groups = json.loads(cleaned)

                # VALIDATE
                flat_ids = [i for grp in groups for i in grp]
                expected_ids = list(range(total_segs))
                
                if flat_ids != expected_ids:
                    print(f"    📝 Raw (200 ký tự cuối): ...{response_text[-200:]}")
                    missing = sorted(set(expected_ids) - set(flat_ids))
                    dupes = sorted({i for i in flat_ids if flat_ids.count(i) > 1})
                    out_of_range = sorted({i for i in flat_ids if i < 0 or i >= total_segs})
                    # Kiểm tra trường hợp đủ chỉ số nhưng sai thứ tự / nhảy cóc
                    order_broken = (not missing and not dupes and not out_of_range)
                
                    print(f"    🔍 DEBUG {filename}: tổng {len(flat_ids)}/{total_segs}")
                    print(f"       - Thiếu: {missing}")
                    print(f"       - Trùng: {dupes}")
                    print(f"       - Ngoài phạm vi: {out_of_range}")
                    print(f"       - Sai thứ tự: {order_broken}")
                    for i in missing:
                        print(f"       - Câu thiếu [{i}]: {flat_segments[i]['text'].strip()[:100]}")
                
                    raise ValueError(
                        f"Validate thất bại: AI trả về {len(flat_ids)} chỉ số, không khớp {total_segs} câu gốc."
                    )

                # ==============
                # REBUILD EVENTS
                # ==============
                built_events = []
                for grp in groups:
                    # Tạo bản sao hoàn toàn mới cho từng segment
                    segs_in_group = [copy.deepcopy(flat_segments[i]) for i in grp]

                    # 1. Cập nhật lại text cho từng segment bản sao
                    for seg in segs_in_group:
                        seg["text"] = normalize_segment_text(seg["text"])

                    # 2. Tạo event mới
                    built_events.append({
                        "event_id": segs_in_group[0]["id"],
                        "start": segs_in_group[0]["start"],
                        "end": segs_in_group[-1]["end"],
                        "text": " ".join(s["text"] for s in segs_in_group),
                        "segments": segs_in_group
                    })

                new_events = built_events
                success = True

                time.sleep(DELAY_BETWEEN_CALLS)

            except Exception as e:
                retry_count += 1
                print(f"    ⚠️ Lỗi file {filename} (Lần thử {retry_count}/{max_retries}): {e}")
                if retry_count < max_retries:
                    backoff = retry_count * 10.0
                    print(f"    💤 Nghỉ {backoff}s rồi thử lại...")
                    time.sleep(backoff)
                else:
                    print("    ❌ SAFE-FALLBACK: Giữ nguyên events gốc cho file này!")
                    new_events = old_events

        data["events"] = new_events

        # Remove để tránh duplicate dữ liệu không cần thiết.
        data.pop("segments", None)

        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

        print(f"    ✅ Đã xử lý xong: {len(old_events)} -> {len(new_events)} events.")

    print(f"Finished fix_event_grouping_task. Tổng {total_part_files} files đã xử lý.")
    kbench.assertions.assert_true(True, expectation=f"Done with {total_part_files} files.")


# Run the task with the specified LLM
fix_event_grouping_task.run(llm=kbench.llms["google/gemini-3.5-flash"])